<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Exploratory Data Analysis for Modeling - Solution

---

### About
In this notebook, we'll explore the critical steps of performing Exploratory Data Analysis (EDA) with machine learning modeling in mind. EDA is not just about understanding your data – it's about making informed decisions that will affect your entire modeling pipeline.

### Learning Objectives
We'll use several real-world data sets to explore:
- Why data exploration is crucial for successful modeling.
- Random variables and their distributions.
- How distributions inform our modeling approach.
- Correlation and its role in feature selection.
- Data standardization techniques.
- The critical distinction between correlation and causation.

### Notebook Guide
1. Setup and Data Loading
2. Why Explore Data for Modeling?
3. Random Variables in Data Science
4. Distributions and Modeling Approach
5. Correlation Analysis
6. Data Standardization
7. Correlation vs. Causation
8. Putting It All Together: Comprehensive EDA for Modeling
9. Conclusions and Key Takeaways
10. Practice Exercises

# 1. Setup and Data Loading

First, let's import the necessary libraries and load our data sets. We'll work with multiple data sets to illustrate different concepts.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.datasets import load_iris, make_regression, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, accuracy_score
from warnings import filterwarnings
filterwarnings('ignore')

# Set the plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

Let's load several data sets for our exploration:

In [ ]:
# Load the California Housing data set - a shortened version of the data used in the first part of Aurelion Geron's
# book Hands-On Machine Learning with Scikit-Learn & TensorFlow
housing_df = pd.read_csv('../data/housing-geron.csv')

# Load the Iris data set
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['species'] = iris.target

# Create a synthetic data set with known properties
X, y = make_regression(n_samples=1000, n_features=5, noise=20, random_state=42)
synthetic_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(5)])
synthetic_df['target'] = y

# Inspect the shapes of each DataFrame
print("Data sets loaded successfully!")
print(f"California Housing: {housing_df.shape}")
print(f"Iris: {iris_df.shape}")
print(f"Synthetic: {synthetic_df.shape}")

# 2. Why Explore Data for Modeling?

Before we build any machine learning model, we need to understand our data. Let's explore why this is crucial.

### Exercise 1: Initial Data Exploration

Complete the code below to perform initial exploration of the California Housing data set. This will help us understand what we're working with.

In [ ]:
# Display basic information about the California Housing data set
# Display the first few rows of the data set
print("First 5 rows of the data set:")
print(housing_df.head())

# Display the data types and missing values information
print("\nData set info:")
print(housing_df.info())

# Display summary statistics
print("\nSummary statistics:")
print(housing_df.describe())

### Exercise 2: Data Quality Issues

Let's create a function to identify potential data quality issues that could affect our modeling.

In [ ]:
def identify_data_issues(df):
    """
    Identify potential data quality issues in a dataframe.
    
    Returns:
    - Missing values per column
    - Duplicate rows
    - Potential outliers (values beyond 3 standard deviations)
    - Low variance features
    """
    issues = {}
    
    # Check for missing values
    issues['missing_values'] = df.isnull().sum()
    
    # Check for duplicate rows
    issues['duplicates'] = df.duplicated().sum()
    
    # Identify potential outliers for numerical columns
    outliers = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        mean = df[col].mean()
        std = df[col].std()
        outliers[col] = ((df[col] < mean - 3*std) | (df[col] > mean + 3*std)).sum()
    issues['outliers'] = outliers
    
    # Identify low variance features
    low_variance = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].std() < 0.01:
            low_variance[col] = df[col].std()
    issues['low_variance'] = low_variance
    
    return issues

# Test the function on Housing data
issues = identify_data_issues(housing_df)
print("Data Quality Issues:")
print(f"Missing values:\n{issues['missing_values']}")
print(f"\nDuplicate rows: {issues['duplicates']}")
print(f"\nOutliers per column:\n{issues['outliers']}")
print(f"\nLow variance features:\n{issues['low_variance']}")

### Reflection: Why EDA Matters for Modeling

Based on your initial exploration, answer these questions:

1. What potential problems did you identify that could affect model performance?
2. How might missing values impact your choice of algorithms?
3. Why is identifying outliers important before building a model?

**Answers:**
1. The California Housing data set has several potential issues:
   - Some columns have outliers (for example `total_rooms` and `median_income`)
   - There might be collinearity between features (we'll check this later)
   - Different features have very different scales, which could affect distance-based algorithms

2. Missing values can impact algorithm choice because:
   - Some algorithms (like tree-based methods) can handle missing values natively
   - Linear models typically require complete data or imputation
   - Neural networks usually require complete data
   - The amount and pattern of missingness determines whether deletion or imputation is appropriate

3. Identifying outliers is important because:
   - They can disproportionately influence linear models and pull the regression line
   - They can affect the mean and standard deviation used in standardization
   - They might represent errors, special cases, or important anomalies that need separate treatment
   - Some algorithms (like SVM) are sensitive to outliers, while others (like tree-based methods) are more robust

# 3. Random Variables in Data Science

In data science, we work with random variables - quantities whose values are subject to variation due to chance. Let's explore this concept.

### Exercise 3: Identifying Random Variables

Let's complete the code to categorize variables in our data sets as discrete or continuous.

In [ ]:
def classify_variables(df):
    """
    Classify variables as discrete or continuous.
    
    Rules:
    - Discrete: integers with few unique values (< 10) or object type
    - Continuous: floats or integers with many unique values
    """
    discrete_vars = []
    continuous_vars = []
    
    for col in df.columns:
        # Check dtype and number of unique values
        if df[col].dtype == 'object' or (df[col].dtype in ['int64', 'int32'] and df[col].nunique() < 10):
            discrete_vars.append(col)
        else:
            continuous_vars.append(col)
    
    return discrete_vars, continuous_vars

# Apply to Housing data set
discrete, continuous = classify_variables(housing_df)
print(f"Discrete variables: {discrete}")
print(f"Continuous variables: {continuous}")

# Apply to Iris data set
discrete_iris, continuous_iris = classify_variables(iris_df)
print(f"\nIris - Discrete variables: {discrete_iris}")
print(f"Iris - Continuous variables: {continuous_iris}")

### Exercise 4: Properties of Random Variables

Let's calculate key properties of random variables in our data set.

In [ ]:
# Calculate properties for a specific variable
variable = 'median_house_value'  # California housing prices

# Calculate mean (expected value)
mean_value = housing_df[variable].mean()

# Calculate variance
variance_value = housing_df[variable].var()

# Calculate standard deviation
std_value = housing_df[variable].std()

# Calculate median, 25th, and 75th percentiles
median_value = housing_df[variable].median()
q25 = housing_df[variable].quantile(0.25)
q75 = housing_df[variable].quantile(0.75)

# Display results
print(f"Properties of {variable}:")
print(f"Mean: {mean_value:.2f}")
print(f"Variance: {variance_value:.2f}")
print(f"Standard Deviation: {std_value:.2f}")
print(f"Median: {median_value:.2f}")
print(f"25th Percentile: {q25:.2f}")
print(f"75th Percentile: {q75:.2f}")
print(f"IQR: {q75 - q25:.2f}")

---

To the slides!

---

# 4. Distributions and Modeling Approach

The distribution of variables significantly impacts our choice of modeling approach. Let's explore how.

### Exercise 5: Visualizing Distributions

Let's complete the function to create comprehensive distribution visualizations.

In [ ]:
def visualize_distribution(df, column):
    """
    Create comprehensive visualization of a variable's distribution.
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Create histogram with KDE
    ax = axes[0, 0]
    sns.histplot(df[column], kde=True, ax=ax)
    ax.set_title(f'Histogram with KDE: {column}')
    
    # Create box plot
    ax = axes[0, 1]
    sns.boxplot(y=df[column], ax=ax)
    ax.set_title(f'Box Plot: {column}')
    
    # Create Q-Q plot (normal distribution)
    ax = axes[1, 0]
    stats.probplot(df[column], dist="norm", plot=ax)
    ax.set_title(f'Q-Q Plot: {column}')
    
    # Create violin plot
    ax = axes[1, 1]
    sns.violinplot(y=df[column], ax=ax)
    ax.set_title(f'Violin Plot:{column}')
    
    plt.suptitle(f'Distribution Analysis: {column}', fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize some variables
visualize_distribution(housing_df, 'median_house_value')
visualize_distribution(housing_df, 'housing_median_age')

### Exercise 6: Testing for Distribution Types

Let's test if variables follow specific distributions using statistical tests.

In [ ]:
def test_distribution(data):
    """
    Test if data follows normal distribution.
    """
    # Perform Shapiro-Wilk test for normality
    stat, p_value = stats.shapiro(data)
    
    # Calculate skewness
    skewness = stats.skew(data)
    
    # Calculate kurtosis
    kurtosis = stats.kurtosis(data)
    
    return {
        'shapiro_statistic': stat,
        'shapiro_p_value': p_value,
        'is_normal': p_value > 0.05,
        'skewness': skewness,
        'kurtosis': kurtosis
    }

# Test some variables
price_test = test_distribution(housing_df['median_house_value'])
print("Price distribution test:", price_test)

age_test = test_distribution(housing_df['housing_median_age'])
print("Age distribution test:", age_test)

# Test a normal-like variable from synthetic data
synthetic_test = test_distribution(synthetic_df['feature_0'])
print("Synthetic feature_0 test:", synthetic_test)

### Exercise 7: Impact on Model Performance

Let's see how distribution affects model performance by comparing models on original and transformed data.

In [ ]:
def compare_models_with_transformation(X, y):
    """
    Compare model performance with and without log transformation.
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Model 1: Original data
    model_original = LinearRegression()
    model_original.fit(X_train, y_train)
    y_pred_original = model_original.predict(X_test)
    r2_original = r2_score(y_test, y_pred_original)
    
    # Model 2: Log-transformed target (for positive targets only)
    if np.all(y_train > 0):
        # Apply log transformation to target
        y_train_log = np.log(y_train)
        y_test_log = np.log(y_test)
        
        # Train model on log-transformed target
        model_log = LinearRegression()
        model_log.fit(X_train, y_train_log)
        y_pred_log = model_log.predict(X_test)
        
        # Transform predictions back to original scale
        y_pred_original_scale = np.exp(y_pred_log)
        r2_log = r2_score(y_test, y_pred_original_scale)
    else:
        r2_log = None
    
    return {
        'r2_original': r2_original,
        'r2_log_transformed': r2_log
    }

# Test on Housing data
X = housing_df.drop(['median_house_value', 'ocean_proximity', 'total_bedrooms'], axis=1)
y = housing_df['median_house_value']
results = compare_models_with_transformation(X, y)
print("Model comparison results:")
print(f"R² with original data: {results['r2_original']:.4f}")
print(f"R² with log-transformed target: {results['r2_log_transformed']:.4f}")

# 5. Correlation Analysis

Understanding the relationships between variables is crucial for feature selection and model interpretation.

### Exercise 8: Correlation Matrix and Heatmap

Create a comprehensive correlation analysis for the California Housing data set.

In [ ]:
def correlation_analysis(df, target_column=None):
    """
    Perform comprehensive correlation analysis.
    """
    # Calculate correlation matrix
    corr_matrix = df.corr()
    
    # Create correlation heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix Heatmap')
    plt.tight_layout()
    plt.show()
    
    if target_column:
        # Find features most correlated with target
        target_corr = corr_matrix[target_column].sort_values(ascending=False)
        
        # Create bar plot of correlations with target
        plt.figure(figsize=(10, 6))
        target_corr[1:].plot(kind='bar')  # Exclude self-correlation
        plt.title(f'Feature Correlations with {target_column}')
        plt.xlabel('Features')
        plt.ylabel('Correlation Coefficient')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    # Identify multicollinearity (correlation > 0.8 between features)
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.8:
                high_corr_pairs.append((corr_matrix.index[i], 
                                      corr_matrix.columns[j], 
                                      corr_matrix.iloc[i, j]))
    
    print("\nHighly correlated feature pairs (|correlation| > 0.8):")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"{feat1} - {feat2}: {corr:.3f}")
    
    return corr_matrix, high_corr_pairs

# Perform analysis
corr_df = housing_df.drop(['ocean_proximity', 'total_bedrooms'], axis=1)
corr_matrix, multicollinear_pairs = correlation_analysis(corr_df, 'median_house_value')

### Exercise 9: Different Types of Correlation

Let's compare Pearson, Spearman, and Kendall correlations to understand their differences.

In [ ]:
def compare_correlation_methods(x, y):
    """
    Compare different correlation methods.
    """
    # Calculate Pearson correlation
    pearson_corr, pearson_p = stats.pearsonr(x, y)
    
    # Calculate Spearman correlation
    spearman_corr, spearman_p = stats.spearmanr(x, y)
    
    # Calculate Kendall correlation
    kendall_corr, kendall_p = stats.kendalltau(x, y)
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Pearson
    axes[0].scatter(x, y, alpha=0.6)
    axes[0].set_title(f'Pearson r = {pearson_corr:.3f}\n(p = {pearson_p:.3f})')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')
    
    # Spearman
    axes[1].scatter(x, y, alpha=0.6)
    axes[1].set_title(f'Spearman ρ = {spearman_corr:.3f}\n(p = {spearman_p:.3f})')
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Y')
    
    # Kendall
    axes[2].scatter(x, y, alpha=0.6)
    axes[2].set_title(f'Kendall τ = {kendall_corr:.3f}\n(p = {kendall_p:.3f})')
    axes[2].set_xlabel('X')
    axes[2].set_ylabel('Y')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'pearson': (pearson_corr, pearson_p),
        'spearman': (spearman_corr, spearman_p),
        'kendall': (kendall_corr, kendall_p)
    }

# Create data with non-linear relationship
x = np.linspace(0, 10, 100)
y = x**2 + np.random.normal(0, 10, 100)

# Compare correlations
correlations = compare_correlation_methods(x, y)
print("Correlation comparison for quadratic relationship:")
for method, (corr, p) in correlations.items():
    print(f"{method}: {corr:.3f} (p={p:.3f})")

### Exercise 10: Correlation's Limitations

Let's create examples where correlation fails to capture relationships.

In [ ]:
def correlation_limitations():
    """
    Demonstrate limitations of correlation.
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Example 1: Quadratic relationship
    x1 = np.linspace(-3, 3, 100)
    y1 = x1**2 + np.random.normal(0, 0.5, 100)
    corr1, _ = stats.pearsonr(x1, y1)
    ax = axes[0, 0]
    ax.scatter(x1, y1, alpha=0.6)
    ax.set_title(f'Quadratic: r = {corr1:.3f}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    # Example 2: Sinusoidal relationship
    x2 = np.linspace(0, 4*np.pi, 100)
    y2 = np.sin(x2) + np.random.normal(0, 0.2, 100)
    corr2, _ = stats.pearsonr(x2, y2)
    ax = axes[0, 1]
    ax.scatter(x2, y2, alpha=0.6)
    ax.set_title(f'Sinusoidal: r = {corr2:.3f}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    # Example 3: Outlier effect
    x3 = np.random.normal(0, 1, 50)
    y3 = x3 + np.random.normal(0, 0.5, 50)
    # Add outliers
    x3 = np.append(x3, [4, 4.5])
    y3 = np.append(y3, [-2, -2.5])
    corr3, _ = stats.pearsonr(x3, y3)
    ax = axes[1, 0]
    ax.scatter(x3, y3, alpha=0.6)
    ax.set_title(f'With Outliers: r = {corr3:.3f}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    # Example 4: Clustered data
    x4 = np.concatenate([np.random.normal(-2, 0.5, 50), np.random.normal(2, 0.5, 50)])
    y4 = np.concatenate([np.random.normal(2, 0.5, 50), np.random.normal(-2, 0.5, 50)])
    corr4, _ = stats.pearsonr(x4, y4)
    ax = axes[1, 1]
    ax.scatter(x4, y4, alpha=0.6)
    ax.set_title(f'Clustered Data: r = {corr4:.3f}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    plt.tight_layout()
    plt.show()

# Demonstrate limitations
correlation_limitations()

---

To the slides!

---

# 6. Data Standardization

Many machine learning algorithms require standardized input. Let's explore different standardization methods.

### Exercise 11: Implementing Standardization Methods

Implement different standardization techniques and visualize their effects.

In [ ]:
def compare_scaling_methods(data):
    """
    Compare different scaling methods.
    """
    # Create scalers
    scalers = {
        'Original': None,
        'StandardScaler': StandardScaler(),
        'MinMaxScaler': MinMaxScaler(),
        'RobustScaler': RobustScaler()
    }
    
    # Apply scalers
    scaled_data = {}
    for name, scaler in scalers.items():
        if name == 'Original':
            scaled_data[name] = data
        else:
            scaled_data[name] = scaler.fit_transform(data)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for i, (name, data_scaled) in enumerate(scaled_data.items()):
        ax = axes[i]
        ax.hist(data_scaled, bins=30, edgecolor='black', alpha=0.7)
        ax.set_title(f'{name}')
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
        
        # Add statistics
        mean = np.mean(data_scaled)
        std = np.std(data_scaled)
        ax.axvline(mean, color='red', linestyle='--', label=f'Mean: {mean:.3f}')
        ax.text(0.02, 0.95, f'Std: {std:.3f}', transform=ax.transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Create box plots
    plt.figure(figsize=(12, 6))
    positions = range(len(scaled_data))
    data_to_plot = []
    labels = []
    
    for name, data_scaled in scaled_data.items():
        data_to_plot.append(data_scaled.flatten())
        labels.append(name)
    
    plt.boxplot(data_to_plot, positions=positions, labels=labels)
    plt.ylabel('Value')
    plt.title('Comparison of Scaling Methods')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Apply to a feature from Housing data set
feature_data = housing_df['median_house_value'].values.reshape(-1, 1)
compare_scaling_methods(feature_data)

### Exercise 12: Impact of Standardization on Model Performance

Let's see how standardization affects different machine learning algorithms.

In [ ]:
def standardization_impact(X, y):
    """
    Compare model performance with and without standardization.
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    results = {}
    
    # Test without standardization
    lr_no_scale = LogisticRegression(max_iter=1000)
    lr_no_scale.fit(X_train, y_train)
    y_pred_no_scale = lr_no_scale.predict(X_test)
    results['logistic_no_scale'] = accuracy_score(y_test, y_pred_no_scale)
    
    # Test with standardization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    lr_with_scale = LogisticRegression(max_iter=1000)
    lr_with_scale.fit(X_train_scaled, y_train)
    y_pred_with_scale = lr_with_scale.predict(X_test_scaled)
    results['logistic_with_scale'] = accuracy_score(y_test, y_pred_with_scale)
    
    # Visualize coefficient magnitudes
    plt.figure(figsize=(12, 6))
    
    x_pos = np.arange(X.shape[1])
    width = 0.35
    
    plt.bar(x_pos - width/2, np.abs(lr_no_scale.coef_[0]), width, 
            label='Without Scaling', alpha=0.7)
    plt.bar(x_pos + width/2, np.abs(lr_with_scale.coef_[0]), width, 
            label='With Scaling', alpha=0.7)
    
    plt.xlabel('Feature Index')
    plt.ylabel('Absolute Coefficient Value')
    plt.title('Logistic Regression Coefficients: With vs Without Scaling')
    plt.legend()
    plt.xticks(x_pos)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return results

# Create classification data set
X_class, y_class = make_classification(n_samples=1000, n_features=10, 
                                      n_informative=5, random_state=42)
results = standardization_impact(X_class, y_class)
print("Model Performance:")
print(f"Without scaling: {results['logistic_no_scale']:.3f}")
print(f"With scaling: {results['logistic_with_scale']:.3f}")

# 7. Correlation vs. Causation

One of the most important distinctions in data science is between correlation and causation.

### Exercise 13: Spurious Correlations

Let's create examples of spurious correlations to understand the difference between correlation and causation.

In [ ]:
def create_spurious_correlation():
    """
    Create examples of spurious correlations.
    """
    # Generate time series data
    time = np.arange(0, 20, 0.1)
    
    # Example 1: Ice cream sales and drowning incidents
    # Both increase in summer due to temperature
    temperature = 20 + 10 * np.sin(2 * np.pi * time / 12) + np.random.normal(0, 2, len(time))
    ice_cream_sales = 100 + 50 * temperature + np.random.normal(0, 20, len(time))
    drowning_incidents = 5 + 2 * temperature + np.random.normal(0, 5, len(time))
    
    # Calculate correlation
    corr_ice_drowning, _ = stats.pearsonr(ice_cream_sales, drowning_incidents)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot ice cream sales vs time
    ax = axes[0, 0]
    ax.plot(time, ice_cream_sales, label='Ice Cream Sales')
    ax.set_xlabel('Time (months)')
    ax.set_ylabel('Ice Cream Sales')
    ax.set_title('Ice Cream Sales Over Time')
    ax.legend()
    
    # Plot drowning incidents vs time
    ax = axes[0, 1]
    ax.plot(time, drowning_incidents, label='Drowning Incidents', color='red')
    ax.set_xlabel('Time (months)')
    ax.set_ylabel('Drowning Incidents')
    ax.set_title('Drowning Incidents Over Time')
    ax.legend()
    
    # Plot ice cream sales vs drowning incidents
    ax = axes[1, 0]
    ax.scatter(ice_cream_sales, drowning_incidents, alpha=0.6)
    ax.set_xlabel('Ice Cream Sales')
    ax.set_ylabel('Drowning Incidents')
    ax.set_title(f'Ice Cream vs Drowning\nCorrelation: {corr_ice_drowning:.3f}')
    
    # Plot temperature vs time (the true cause)
    ax = axes[1, 1]
    ax.plot(time, temperature, label='Temperature', color='green')
    ax.set_xlabel('Time (months)')
    ax.set_ylabel('Temperature')
    ax.set_title('Temperature Over Time (True Cause)')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    return corr_ice_drowning

# Create spurious correlation example
correlation = create_spurious_correlation()
print(f"Correlation between ice cream sales and drowning: {correlation:.3f}")
print("But this is a spurious correlation! Both are caused by temperature.")

### Exercise 14: Confounding Variables

Let's create a scenario with a confounding variable to understand how correlation can be misleading.

In [ ]:
def demonstrate_confounding():
    """
    Demonstrate the effect of confounding variables.
    """
    np.random.seed(42)
    n_samples = 1000
    
    # Confounding variable: Age
    age = np.random.uniform(18, 80, n_samples)
    
    # Coffee consumption increases with age
    coffee_consumption = 0.5 + 0.05 * age + np.random.normal(0, 1, n_samples)
    coffee_consumption = np.maximum(0, coffee_consumption)  # No negative values
    
    # Health problems increase with age
    health_problems = 0.01 * age**1.5 + np.random.normal(0, 2, n_samples)
    health_problems = np.maximum(0, health_problems)  # No negative values
    
    # Calculate correlations
    corr_coffee_health, _ = stats.pearsonr(coffee_consumption, health_problems)
    corr_age_coffee, _ = stats.pearsonr(age, coffee_consumption)
    corr_age_health, _ = stats.pearsonr(age, health_problems)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Plot 1: Coffee vs Health Problems
    axes[0, 0].scatter(coffee_consumption, health_problems, alpha=0.5)
    axes[0, 0].set_xlabel('Coffee Consumption')
    axes[0, 0].set_ylabel('Health Problems')
    axes[0, 0].set_title(f'Coffee vs Health\nCorrelation: {corr_coffee_health:.3f}')
    
    # Plot 2: Age vs Coffee
    axes[0, 1].scatter(age, coffee_consumption, alpha=0.5)
    axes[0, 1].set_xlabel('Age')
    axes[0, 1].set_ylabel('Coffee Consumption')
    axes[0, 1].set_title(f'Age vs Coffee\nCorrelation: {corr_age_coffee:.3f}')
    
    # Plot 3: Age vs Health Problems
    axes[1, 0].scatter(age, health_problems, alpha=0.5)
    axes[1, 0].set_xlabel('Age')
    axes[1, 0].set_ylabel('Health Problems')
    axes[1, 0].set_title(f'Age vs Health\nCorrelation: {corr_age_health:.3f}')
    
    # Plot 4: 3D visualization showing all three variables
    axes[1, 1].remove()  # Remove to make space for colorbar
    ax = fig.add_subplot(224, projection='3d')
    scatter = ax.scatter(age, coffee_consumption, health_problems, 
                        c=age, cmap='viridis', alpha=0.6)
    ax.set_xlabel('Age')
    ax.set_ylabel('Coffee Consumption')
    ax.set_zlabel('Health Problems')
    ax.set_title('3D View: Age, Coffee, Health')
    plt.colorbar(scatter, ax=ax, label='Age')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'coffee_health_correlation': corr_coffee_health,
        'age_coffee_correlation': corr_age_coffee,
        'age_health_correlation': corr_age_health
    }

# Demonstrate confounding
correlations = demonstrate_confounding()
print("Correlation Analysis:")
print(f"Coffee vs Health: {correlations['coffee_health_correlation']:.3f}")
print(f"Age vs Coffee: {correlations['age_coffee_correlation']:.3f}")
print(f"Age vs Health: {correlations['age_health_correlation']:.3f}")
print("\nConclusion: Coffee appears correlated with health problems,")
print("but age is the confounding variable causing both!")

### Exercise 15: Exploring Correlation in Real Data

Let's examine potential spurious correlations in our real data sets.

In [ ]:
def explore_correlations_critically(df):
    """
    Critically examine correlations in real data.
    """
    # Find the highest correlations
    corr_matrix = df.corr()
    
    # Find the top 5 highest correlations (excluding diagonal)
    high_correlations = []
    
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            correlation = corr_matrix.iloc[i, j]
            high_correlations.append({
                'var1': corr_matrix.index[i],
                'var2': corr_matrix.columns[j],
                'correlation': correlation
            })
    
    # Sort by absolute correlation
    high_correlations.sort(key=lambda x: abs(x['correlation']), reverse=True)
    top_5 = high_correlations[:5]
    
    # Visualize the top correlations
    fig, axes = plt.subplots(3, 2, figsize=(15, 18))
    axes = axes.flatten()
    
    for idx, corr_info in enumerate(top_5):
        var1 = corr_info['var1']
        var2 = corr_info['var2']
        correlation = corr_info['correlation']
        
        ax = axes[idx]
        ax.scatter(df[var1], df[var2], alpha=0.6)
        ax.set_xlabel(var1)
        ax.set_ylabel(var2)
        ax.set_title(f'{var1} vs {var2}\nCorrelation: {correlation:.3f}')
        
        # Add regression line
        z = np.polyfit(df[var1], df[var2], 1)
        p = np.poly1d(z)
        ax.plot(df[var1], p(df[var1]), "r--", alpha=0.8)
    
    # Remove the unused subplot
    fig.delaxes(axes[5])
    
    plt.tight_layout()
    plt.show()
    
    return top_5

# Explore California Housing data set
corr_df = housing_df.drop(['ocean_proximity', 'total_bedrooms'], axis=1)
high_corr = explore_correlations_critically(corr_df)
print("Top correlations in California Housing data set:")
for i, corr_info in enumerate(high_corr, 1):
    print(f"{i}. {corr_info['var1']} vs {corr_info['var2']}: {corr_info['correlation']:.3f}")

# Critical analysis
print("\nCritical Analysis:")
print("1. RAD-TAX (0.910): Likely causal. Tax rates and accessibility to highways are policy-related.")
print("2. NOX-DIS (-0.769): Likely causal. Air pollution decreases with distance from employment centers.")
print("3. INDUS-NOX (0.764): Likely causal. Industrial areas tend to have more pollution.")
print("4. AGE-DIS (-0.748): Likely causal. Older housing tends to be closer to city centers.")
print("5. LSTAT-MEDV (-0.738): Could be causal. Lower status population areas typically have lower home values.")

### Reflection: Correlation vs. Causation

Based on your exploration, answer these questions:

1. What are three examples of correlations that don't imply causation?
2. How can confounding variables mislead our analysis?
3. When building predictive models, when do we need causation vs. when is correlation sufficient?

**Answers:**
1. Three examples of correlations without causation:
   - Ice cream sales and drowning incidents (both caused by summer/heat)
   - Coffee consumption and health problems (both caused by age)
   - Number of firefighters at a scene and damage caused (both caused by fire severity)

2. Confounding variables can mislead analysis by:
   - Creating apparent relationships between variables that aren't directly connected
   - Hiding the true causal mechanism
   - Leading to incorrect conclusions about interventions (e.g., reducing coffee won't improve health)
   - Biasing model predictions if the confounding relationship changes

3. Correlation vs. causation in predictive modeling:
   - Correlation is sufficient for: Pure prediction tasks where we don't need to understand why
   - Causation is needed for: Policy decisions, interventions, understanding mechanisms
   - Example: Predicting house prices (correlation sufficient) vs. deciding which factors to change to increase house values (causation needed)

# 8. Putting It All Together: Comprehensive EDA for Modeling

Let's create a comprehensive EDA pipeline that incorporates everything we've learned.

In [ ]:
def comprehensive_eda(df, target_column):
    """
    Perform comprehensive EDA for machine learning.
    """
    report = {}
    
    # 1. Basic data properties
    report['shape'] = df.shape
    report['missing_values'] = df.isnull().sum().to_dict()
    report['duplicates'] = df.duplicated().sum()
    
    # 2. Variable types and distributions
    discrete_vars, continuous_vars = classify_variables(df)
    report['variable_types'] = {
        'discrete': discrete_vars,
        'continuous': continuous_vars
    }
    
    # Test distributions for continuous variables
    distribution_tests = {}
    for var in continuous_vars:
        distribution_tests[var] = test_distribution(df[var])
    report['distribution_tests'] = distribution_tests
    
    # 3. Correlation analysis
    if target_column in df.columns:
        corr_matrix = df.corr()
        target_correlations = corr_matrix[target_column].sort_values(ascending=False)
        report['target_correlations'] = target_correlations.to_dict()
        
        # Find multicollinear features
        multicollinear_features = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.8:
                    multicollinear_features.append(
                        (corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.iloc[i, j])
                    )
        report['multicollinear_features'] = multicollinear_features
    
    # 4. Outlier detection
    outliers = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outlier_mask = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
        outliers[col] = outlier_mask.sum()
    report['outliers'] = outliers
    
    # 5. Feature importance (basic)
    if target_column in df.columns:
        # Simple variance-based importance
        feature_importance = {}
        for col in df.select_dtypes(include=[np.number]).columns:
            if col != target_column:
                correlation = abs(df[col].corr(df[target_column]))
                feature_importance[col] = correlation
        
        report['feature_importance'] = dict(sorted(feature_importance.items(), 
                                                  key=lambda x: x[1], reverse=True))
    
    # 6. Recommendations
    recommendations = []
    
    # Missing values
    if sum(report['missing_values'].values()) > 0:
        recommendations.append("Handle missing values - consider imputation or removal")
    
    # Outliers
    high_outlier_cols = [col for col, count in outliers.items() if count > len(df) * 0.05]
    if high_outlier_cols:
        recommendations.append(f"Address outliers in: {', '.join(high_outlier_cols)}")
    
    # Skewed distributions
    skewed_vars = [var for var, test in distribution_tests.items() 
                   if abs(test['skewness']) > 1]
    if skewed_vars:
        recommendations.append(f"Consider transforming skewed variables: {', '.join(skewed_vars)}")
    
    # Multicollinearity
    if report.get('multicollinear_features'):
        recommendations.append("Address multicollinearity - consider removing correlated features")
    
    # Scaling
    recommendations.append("Apply feature scaling for distance-based algorithms")
    
    report['recommendations'] = recommendations
    
    return report

# Apply to Housing data set
eda_df = housing_df.drop(['ocean_proximity', 'total_bedrooms'], axis=1)
eda_report = comprehensive_eda(eda_df, 'median_house_value')

# Display report
print("=== EDA Report for California Housing Data set ===")
print(f"\nData set shape: {eda_report['shape']}")
print(f"Duplicate rows: {eda_report['duplicates']}")

print("\nTop 5 features correlated with PRICE:")
for i, (feature, corr) in enumerate(list(eda_report['target_correlations'].items())[1:6], 1):
    print(f"{i}. {feature}: {corr:.3f}")

print("\nMulticollinear feature pairs:")
for feat1, feat2, corr in eda_report['multicollinear_features']:
    print(f"{feat1} - {feat2}: {corr:.3f}")

print("\nOutliers per column:")
for col, count in eda_report['outliers'].items():
    if count > 0:
        print(f"{col}: {count} outliers ({count/len(eda_df)*100:.1f}%)")

print("\nRecommendations:")
for i, rec in enumerate(eda_report['recommendations'], 1):
    print(f"{i}. {rec}")

# 9. Conclusions and Key Takeaways

Let's summarize what we've learned about EDA for machine learning:

### Key Takeaways

1. **Why explore data before modeling?**
   - Identify data quality issues (missing values, outliers, duplicates)
   - Understand variable distributions and relationships
   - Inform feature engineering and model selection decisions
   - Prevent costly mistakes and wasted modeling efforts

2. **Random variables in data science:**
   - Variables whose values are subject to chance
   - Can be discrete (countable) or continuous (measurable)
   - Characterized by distribution properties (mean, variance, skewness)

3. **How distributions affect modeling:**
   - Linear models assume normally distributed residuals
   - Skewed distributions may require transformations
   - Tree-based models are more robust to distribution shapes
   - Outliers affect different algorithms differently

4. **Correlation's role and limitations:**
   - Measures linear relationships between variables
   - Useful for feature selection and understanding relationships
   - Doesn't capture non-linear relationships
   - Sensitive to outliers

5. **When and why to standardize data:**
   - Required for distance-based algorithms (KNN, SVM)
   - Helps with gradient descent convergence
   - Makes feature coefficients comparable
   - Different methods for different data characteristics

6. **Correlation vs. Causation:**
   - Correlation doesn't imply causation
   - Confounding variables can create spurious correlations
   - Causal inference requires additional methods/experiments
   - Important for decision-making vs. pure prediction

# 10. Practice Exercises

Try these exercises to reinforce your learning:

### Exercise A: Your Own Data

1. Load a data set of your choice
2. Perform comprehensive EDA
3. Identify the key insights that would inform your modeling approach
4. Document your findings

In [ ]:
# Example solution with the Iris data set
iris_eda = comprehensive_eda(iris_df, 'species')
print("EDA Report for Iris Data set:")
print(f"Shape: {iris_eda['shape']}")
print(f"Variable types: {iris_eda['variable_types']}")
print("\nKey insights:")
print("1. No missing values or duplicates - clean data set")
print("2. All features are continuous measurements")
print("3. Target is categorical (species) - classification problem")
print("4. Some features show strong correlation (petal length and width)")

### Exercise B: Challenge - Misleading Correlations

Create a synthetic data set where:
1. Two variables are highly correlated
2. The correlation is due to a hidden third variable
3. A model based on the correlation would make poor predictions

Document your example and explain the lesson it teaches.

In [ ]:
# Solution: School performance example
np.random.seed(42)
n_students = 1000

# Hidden variable: Family income
family_income = np.random.exponential(50000, n_students)

# Hours of private tutoring (increases with income)
tutoring_hours = np.maximum(0, 0.0001 * family_income + np.random.normal(0, 2, n_students))

# Test scores (influenced by income through various mechanisms)
test_scores = 50 + 0.0003 * family_income + np.random.normal(0, 10, n_students)
test_scores = np.clip(test_scores, 0, 100)

# Calculate correlation
corr_tutoring_scores, _ = stats.pearsonr(tutoring_hours, test_scores)

# Visualization
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.scatter(tutoring_hours, test_scores, alpha=0.5)
plt.xlabel('Tutoring Hours')
plt.ylabel('Test Scores')
plt.title(f'Tutoring vs Scores\nCorr: {corr_tutoring_scores:.3f}')

plt.subplot(1, 3, 2)
plt.scatter(family_income, tutoring_hours, alpha=0.5)
plt.xlabel('Family Income')
plt.ylabel('Tutoring Hours')
plt.title('Income vs Tutoring')

plt.subplot(1, 3, 3)
plt.scatter(family_income, test_scores, alpha=0.5)
plt.xlabel('Family Income')
plt.ylabel('Test Scores')
plt.title('Income vs Scores')

plt.tight_layout()
plt.show()

print(f"Correlation between tutoring hours and test scores: {corr_tutoring_scores:.3f}")
print("\nLesson: Simply increasing tutoring hours may not improve scores")
print("if the underlying cause (family resources/environment) isn't addressed.")
print("Policy based on this correlation alone would be ineffective.")

### Final Reflection

Write a brief paragraph (3-5 sentences) explaining how the concepts in this notebook will change your approach to machine learning projects:

**Reflection:**

Understanding the principles of EDA for modeling has fundamentally changed my approach to machine learning projects. Rather than rushing to apply algorithms, I now recognize the critical importance of thoroughly exploring data distributions, relationships, and quality issues first. The distinction between correlation and causation has made me more cautious about interpreting model results and making recommendations, especially for interventions. I've learned that the choice of preprocessing techniques (like scaling and transformation) should be informed by both the data characteristics and the algorithm requirements. Most importantly, I now see EDA not as a preliminary step but as an integral part of the modeling process that directly impacts model performance and interpretability.